# Prepare four climate-sensitivity MasterInput databases

This notebook creates four independent copies of `MasterInput.db` for the AgriScale training exercise. The historical database is unchanged. The perturbations are deliberately simple sensitivity scenarios, not CMIP6/SSP projections.

- `HIST`: unchanged climate
- `TPLUS2`: `tmin`, `tmax`, and `tmoy` increased by 2 °C
- `RAIN80`: daily rainfall multiplied by 0.80
- `TPLUS2_RAIN80`: both perturbations

Relative humidity, dew point, radiation, wind, and surface pressure are intentionally left unchanged.

In [ ]:
from contextlib import closing
from datetime import datetime, timezone
from pathlib import Path
import sqlite3

SOURCE_DB = Path("/home/midingoyi/datamill/data_Maroc/MasterInput.db")
OUTPUT_DIR = SOURCE_DB.parent
OVERWRITE = False  # Change to True only when intentionally regenerating outputs.

SCENARIOS = {
    "HIST": {"temperature_delta_c": 0.0, "rain_factor": 1.0},
    "TPLUS2": {"temperature_delta_c": 2.0, "rain_factor": 1.0},
    "RAIN80": {"temperature_delta_c": 0.0, "rain_factor": 0.80},
    "TPLUS2_RAIN80": {"temperature_delta_c": 2.0, "rain_factor": 0.80},
}

assert SOURCE_DB.is_file(), f"Source database not found: {SOURCE_DB}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DB, OUTPUT_DIR

In [ ]:
def connect_read_only(path: Path) -> sqlite3.Connection:
    return sqlite3.connect(f"file:{path.resolve()}?mode=ro", uri=True)


def table_columns(connection: sqlite3.Connection, table: str) -> set[str]:
    return {row[1].lower() for row in connection.execute(f'PRAGMA table_info("{table}")')}


def validate_source(path: Path) -> dict:
    required = {"tmin", "tmax", "tmoy", "rain"}
    with closing(connect_read_only(path)) as connection:
        integrity = connection.execute("PRAGMA integrity_check").fetchone()[0]
        columns = table_columns(connection, "RAClimateD")
        missing = required - columns
        if integrity != "ok":
            raise RuntimeError(f"SQLite integrity check failed: {integrity}")
        if missing:
            raise RuntimeError(f"RAClimateD is missing columns: {sorted(missing)}")
        count, first_year, last_year = connection.execute(
            "SELECT COUNT(*), MIN(year), MAX(year) FROM RAClimateD"
        ).fetchone()
    return {"rows": count, "first_year": first_year, "last_year": last_year}


source_summary = validate_source(SOURCE_DB)
source_summary

In [ ]:
def sqlite_backup(source: Path, target: Path) -> None:
    if target.exists():
        if not OVERWRITE:
            raise FileExistsError(
                f"Refusing to overwrite {target}. Set OVERWRITE=True to regenerate."
            )
        target.unlink()
    with closing(connect_read_only(source)) as source_connection:
        with closing(sqlite3.connect(target)) as target_connection:
            source_connection.backup(target_connection)


def prepare_scenario(name: str, settings: dict) -> dict:
    target = OUTPUT_DIR / f"MasterInput_{name}.db"
    sqlite_backup(SOURCE_DB, target)

    temperature_delta = float(settings["temperature_delta_c"])
    rain_factor = float(settings["rain_factor"])
    created_at = datetime.now(timezone.utc).isoformat()

    with closing(sqlite3.connect(target)) as connection:
        connection.execute("BEGIN IMMEDIATE")
        if temperature_delta:
            connection.execute(
                """
                UPDATE RAClimateD
                SET tmin = tmin + ?, tmax = tmax + ?, tmoy = tmoy + ?
                """,
                (temperature_delta, temperature_delta, temperature_delta),
            )
        if rain_factor != 1.0:
            connection.execute(
                "UPDATE RAClimateD SET rain = MAX(0.0, rain * ?)",
                (rain_factor,),
            )

        # Keep scenario names visible in exported results without changing idPoint.
        connection.execute(
            "UPDATE SimUnitList SET idsim = idsim || ? WHERE idsim NOT LIKE ?",
            (f"__{name}", f"%__{name}"),
        )
        connection.execute("DROP TABLE IF EXISTS ClimateScenarioMetadata")
        connection.execute(
            """
            CREATE TABLE ClimateScenarioMetadata (
                scenario TEXT PRIMARY KEY,
                temperature_delta_c REAL NOT NULL,
                rain_factor REAL NOT NULL,
                source_database TEXT NOT NULL,
                created_at_utc TEXT NOT NULL,
                description TEXT NOT NULL
            )
            """
        )
        connection.execute(
            "INSERT INTO ClimateScenarioMetadata VALUES (?, ?, ?, ?, ?, ?)",
            (
                name, temperature_delta, rain_factor, SOURCE_DB.name, created_at,
                "Pedagogical delta-change sensitivity scenario; not an SSP projection.",
            ),
        )
        connection.commit()
        connection.execute("PRAGMA wal_checkpoint(TRUNCATE)").fetchone()
        journal_mode = connection.execute("PRAGMA journal_mode=DELETE").fetchone()[0]
        if journal_mode.lower() != "delete":
            raise RuntimeError(f"Could not finalize {target} in DELETE journal mode")

        integrity = connection.execute("PRAGMA integrity_check").fetchone()[0]
        count, first_year, last_year, min_rain = connection.execute(
            "SELECT COUNT(*), MIN(year), MAX(year), MIN(rain) FROM RAClimateD"
        ).fetchone()
        simunits = connection.execute("SELECT COUNT(*) FROM SimUnitList").fetchone()[0]

    if integrity != "ok":
        raise RuntimeError(f"Integrity check failed for {target}: {integrity}")
    if count != source_summary["rows"]:
        raise RuntimeError(f"Climate row count changed for {target}: {count}")
    if min_rain is not None and min_rain < 0:
        raise RuntimeError(f"Negative rainfall found in {target}")

    return {
        "scenario": name, "database": str(target), "climate_rows": count,
        "first_year": first_year, "last_year": last_year,
        "simulation_units": simunits, "integrity": integrity,
    }


In [ ]:
results = [prepare_scenario(name, settings) for name, settings in SCENARIOS.items()]
results

In [ ]:
# Compact numerical verification against the historical copy.
def climate_means(path: Path) -> tuple[float, float, float, float]:
    with closing(connect_read_only(path)) as connection:
        return connection.execute(
            "SELECT AVG(tmin), AVG(tmax), AVG(tmoy), AVG(rain) FROM RAClimateD"
        ).fetchone()


hist_means = climate_means(OUTPUT_DIR / "MasterInput_HIST.db")
verification = {}
for name, settings in SCENARIOS.items():
    means = climate_means(OUTPUT_DIR / f"MasterInput_{name}.db")
    verification[name] = {
        "delta_tmin": means[0] - hist_means[0],
        "delta_tmax": means[1] - hist_means[1],
        "delta_tmoy": means[2] - hist_means[2],
        "rain_ratio": means[3] / hist_means[3],
    }
verification

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    %pip install matplotlib
    import matplotlib.pyplot as plt

## Compare historical and modified climates

Each panel shows the monthly climatology over all available years. Bars are mean monthly precipitation totals and the line is mean air temperature. Identical axes make the four scenarios directly comparable.

In [ ]:
import calendar
import matplotlib.pyplot as plt


def monthly_climatology(path: Path) -> dict:
    with closing(connect_read_only(path)) as connection:
        temperatures = dict(connection.execute(
            """
            SELECT Nmonth, AVG(tmoy)
            FROM RAClimateD
            GROUP BY Nmonth
            ORDER BY Nmonth
            """
        ))
        precipitation = dict(connection.execute(
            """
            SELECT Nmonth, AVG(monthly_rain)
            FROM (
                SELECT year, Nmonth, SUM(rain) AS monthly_rain
                FROM RAClimateD
                GROUP BY year, Nmonth
            )
            GROUP BY Nmonth
            ORDER BY Nmonth
            """
        ))
    months = range(1, 13)
    return {
        "temperature": [temperatures[month] for month in months],
        "precipitation": [precipitation[month] for month in months],
    }


climatologies = {
    name: monthly_climatology(OUTPUT_DIR / f"MasterInput_{name}.db")
    for name in SCENARIOS
}
months = list(range(1, 13))
month_labels = [calendar.month_abbr[month] for month in months]
temperature_limits = (
    min(min(values["temperature"]) for values in climatologies.values()) - 1,
    max(max(values["temperature"]) for values in climatologies.values()) + 1,
)
precipitation_limit = max(
    max(values["precipitation"]) for values in climatologies.values()
) * 1.15

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)
rain_axes = []
for axis, (name, values) in zip(axes.flat, climatologies.items()):
    rain_axis = axis.twinx()
    rain_axes.append(rain_axis)
    bars = rain_axis.bar(
        months, values["precipitation"], color="#4C78A8", alpha=0.65,
        label="Monthly precipitation",
    )
    line, = axis.plot(
        months, values["temperature"], color="#D1495B", marker="o",
        linewidth=2.2, label="Mean temperature",
    )
    axis.set_title(name.replace("_", " + "), fontweight="bold")
    axis.set_ylim(*temperature_limits)
    rain_axis.set_ylim(0, precipitation_limit)
    axis.set_xticks(months, month_labels)
    axis.grid(axis="y", alpha=0.25)

for axis in axes[:, 0]:
    axis.set_ylabel("Mean temperature (°C)", color="#D1495B")
for axis in axes.flat:
    axis.set_xlabel("Month")

# Label precipitation axes only on the right column to keep the figure readable.
for rain_axis in (rain_axes[1], rain_axes[3]):
    rain_axis.set_ylabel("Mean monthly precipitation (mm)", color="#4C78A8")

fig.legend([line, bars], ["Mean temperature", "Monthly precipitation"],
           loc="upper center", ncol=2, frameon=False)
fig.suptitle("Historical and perturbed climate scenarios — Tensift, Morocco",
             fontsize=15, y=0.98)
fig.tight_layout(rect=(0, 0, 1, 0.94))
FIGURE_PATH = OUTPUT_DIR.parent / "scripts" / "climate_scenarios_comparison.png"
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH, dpi=200, bbox_inches="tight")
plt.show()
FIGURE_PATH

In [ ]:
# Base paths derived from the repository root, not from the kernel cwd
import os
from modfilegen import GlobalVariables
import sqlite3
from pathlib import Path
import os
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
data_dir = "/data/data_Maroc"


ori_mi = os.path.join(data_dir, "ori_MasterInput.db")
models_dict_db = os.path.join(data_dir, "ModelsDictionaryArise.db")
# Cultivar/plant parameters folder
cultivars_folder_dssat = os.path.join(data_dir, "cultivars", "dssat")
# Configuration
n_threads = 4  # Number of parallel threads
n_parts = 1    # Number of data partitions
dt = 0        # choice to delete intermediate files 0 (no) or 1 (yes)
# Verify files exist
print("Checking files...")
print(f"Models Dict DB exists: {os.path.exists(models_dict_db)}")
print(f"Dssat Cultivars folder exists: {os.path.exists(cultivars_folder_dssat)}")

## HIST

In [ ]:
# Database paths
master_input_db = os.path.join(data_dir, "MasterInput.db")
print(f"Master Input DB exists: {os.path.exists(master_input_db)}")
# Set global variables

GlobalVariables["dbModelsDictionary"] = str(models_dict_db)
GlobalVariables["nthreads"] = n_threads
GlobalVariables["dt"] = dt
GlobalVariables["parts"] = n_parts
GlobalVariables["ori_MI"] = master_input_db
GlobalVariables["pltfolder"] = str(cultivars_folder_dssat)
GlobalVariables["dssat_version"] = "v48"
GlobalVariables["dssat_mode"] = "successive"

from modfilegen.Converter import run_dssat

master_input_db_hist = os.path.join(data_dir, "MasterInput_HIST.db")
output_dir_dssat = os.path.join(data_dir, "output_dssat_hist")
temp_dir_dssat = os.path.join(output_dir_dssat, "temp")
# Create output directories if they don't exist
os.makedirs(output_dir_dssat, exist_ok=True)
os.makedirs(temp_dir_dssat  , exist_ok=True)
GlobalVariables["directorypath"] = str(output_dir_dssat)
GlobalVariables["dbMasterInput"] = str(master_input_db_hist)
GlobalVariables["tempDir"] = str(temp_dir_dssat)

print("Starting Dssat conversion...")
print("=" * 60)

try:
    run_dssat()
    print("\n" + "=" * 60)
    print("✅ Dssat conversion completed successfully!")
except Exception as e:
    print("\n" + "=" * 60)
    print(f"❌ Error during conversion: {e}")
    import traceback
    traceback.print_exc()




## RAIN80

In [ ]:
master_input_db = os.path.join(data_dir, "MasterInput_RAIN80.db")
output_dir_dssat = os.path.join(data_dir, "output_dssat_RAIN80")
temp_dir_dssat = os.path.join(output_dir_dssat, "temp")
# Create output directories if they don't exist
os.makedirs(output_dir_dssat, exist_ok=True)
os.makedirs(temp_dir_dssat  , exist_ok=True)
GlobalVariables["directorypath"] = str(output_dir_dssat)
GlobalVariables["dbMasterInput"] = str(master_input_db)
GlobalVariables["tempDir"] = str(temp_dir_dssat)

print("Starting Dssat conversion...")
print("=" * 60)

try:
    run_dssat()
    print("\n" + "=" * 60)
    print("✅ Dssat conversion completed successfully!")
except Exception as e:
    print("\n" + "=" * 60)
    print(f"❌ Error during conversion: {e}")
    import traceback
    traceback.print_exc()


## TPLUS2

In [ ]:
master_input_db = os.path.join(data_dir, "MasterInput_TPLUS2.db")
output_dir_dssat = os.path.join(data_dir, "output_dssat_TPLUS2")
temp_dir_dssat = os.path.join(output_dir_dssat, "temp")
# Create output directories if they don't exist
os.makedirs(output_dir_dssat, exist_ok=True)
os.makedirs(temp_dir_dssat  , exist_ok=True)
GlobalVariables["directorypath"] = str(output_dir_dssat)
GlobalVariables["dbMasterInput"] = str(master_input_db)
GlobalVariables["tempDir"] = str(temp_dir_dssat)

print("Starting Dssat conversion...")
print("=" * 60)

try:
    run_dssat()
    print("\n" + "=" * 60)
    print("✅ Dssat conversion completed successfully!")
except Exception as e:
    print("\n" + "=" * 60)
    print(f"❌ Error during conversion: {e}")
    import traceback
    traceback.print_exc()

## TPLUS2__RAIN80

In [ ]:
master_input_db = os.path.join(data_dir, "MasterInput_TPLUS2_RAIN80.db")
output_dir_dssat = os.path.join(data_dir, "output_dssat_TPLUS2_RAIN80")
temp_dir_dssat = os.path.join(output_dir_dssat, "temp")
# Create output directories if they don't exist
os.makedirs(output_dir_dssat, exist_ok=True)
os.makedirs(temp_dir_dssat  , exist_ok=True)
GlobalVariables["directorypath"] = str(output_dir_dssat)
GlobalVariables["dbMasterInput"] = str(master_input_db)
GlobalVariables["tempDir"] = str(temp_dir_dssat)

print("Starting Dssat conversion...")
print("=" * 60)

try:
    run_dssat()
    print("\n" + "=" * 60)
    print("✅ Dssat conversion completed successfully!")
except Exception as e:
    print("\n" + "=" * 60)
    print(f"❌ Error during conversion: {e}")
    import traceback
    traceback.print_exc()

## 1. How do yield distributions differ among climate scenarios?

We first compare the distribution of annual simulated grain yield under the four climate configurations. The aim is descriptive: identify changes in central tendency, dispersion, asymmetry, and extreme years before conducting paired inferential tests. Because the same weather years are used in every scenario, later analyses will explicitly retain the year-to-year pairing.

In [ ]:
import csv
from pathlib import Path
from statistics import mean, median

SCENARIO_OUTPUTS = {
    "HIST": Path(data_dir) / "output_dssat_hist",
    "TPLUS2": Path(data_dir) / "output_dssat_TPLUS2",
    "RAIN80": Path(data_dir) / "output_dssat_RAIN80",
    "TPLUS2_RAIN80": Path(data_dir) / "output_dssat_TPLUS2_RAIN80",
}


def latest_successive_result(directory: Path) -> Path:
    candidates = list(directory.glob("*_dssat_successive.csv"))
    if not candidates:
        raise FileNotFoundError(f"No successive DSSAT result found in {directory}")
    return max(candidates, key=lambda path: path.stat().st_mtime)


def read_annual_yields(path: Path) -> dict[int, float]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    annual = {}
    for row in rows:
        year = int(float(row["HYEAR"]))
        yield_kg_ha = float(row["Yield"])
        if yield_kg_ha > -90:
            if year in annual:
                raise ValueError(f"Duplicate harvest year {year} in {path}")
            annual[year] = yield_kg_ha / 1000.0  # Convert to t/ha.
    return annual


result_files = {
    scenario: latest_successive_result(directory)
    for scenario, directory in SCENARIO_OUTPUTS.items()
}
annual_yields = {
    scenario: read_annual_yields(path)
    for scenario, path in result_files.items()
}

reference_years = set(annual_yields["HIST"])
for scenario, values in annual_yields.items():
    if set(values) != reference_years:
        raise ValueError(f"{scenario} does not contain the same harvest years as HIST")

print(f"Paired harvest years: {min(reference_years)}–{max(reference_years)} "
      f"(n={len(reference_years)})")
print(f"{'Scenario':<20} {'Mean':>8} {'Median':>8} {'Minimum':>9} {'Maximum':>9}")
for scenario, values_by_year in annual_yields.items():
    values = list(values_by_year.values())
    print(f"{scenario:<20} {mean(values):8.2f} {median(values):8.2f} "
          f"{min(values):9.2f} {max(values):9.2f}")

result_files

In [ ]:
scenario_order = ["HIST", "TPLUS2", "RAIN80", "TPLUS2_RAIN80"]
scenario_labels = ["Historical", "+2 °C", "Rain −20%", "+2 °C & Rain −20%"]
scenario_colors = ["#4C78A8", "#F58518", "#54A24B", "#E45756"]
boxplot_data = [
    [annual_yields[scenario][year] for year in sorted(reference_years)]
    for scenario in scenario_order
]

fig, axis = plt.subplots(figsize=(11, 6.5))
boxplot = axis.boxplot(
    boxplot_data,
    tick_labels=scenario_labels,
    patch_artist=True,
    showmeans=True,
    meanprops={"marker": "D", "markerfacecolor": "white",
               "markeredgecolor": "black", "markersize": 6},
    medianprops={"color": "black", "linewidth": 2},
)
for patch, color in zip(boxplot["boxes"], scenario_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.65)

# Show all simulated years; fixed offsets avoid random changes between runs.
n_years = len(reference_years)
offsets = [(-0.13 + 0.26 * index / max(1, n_years - 1)) for index in range(n_years)]
for position, (values, color) in enumerate(zip(boxplot_data, scenario_colors), start=1):
    axis.scatter(
        [position + offset for offset in offsets], values,
        s=22, color=color, edgecolor="white", linewidth=0.35, alpha=0.8, zorder=3,
    )

axis.set_ylabel("Simulated grain yield (t ha⁻¹)")
axis.set_title("Distribution of annual rainfed wheat yields by climate scenario")
axis.grid(axis="y", alpha=0.25)
axis.text(
    0.99, 0.98, "Diamond: mean | Line: median | Points: harvest years",
    transform=axis.transAxes, ha="right", va="top", fontsize=9, color="#444444",
)
fig.tight_layout()
YIELD_BOXPLOT_PATH = Path(data_dir).parent / "scripts" / "yield_distributions_boxplot.png"
YIELD_BOXPLOT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(YIELD_BOXPLOT_PATH, dpi=200, bbox_inches="tight")
plt.show()
YIELD_BOXPLOT_PATH

### Interpretation

The historical scenario has the highest central yield (mean 4.36 t ha⁻¹; median 4.52 t ha⁻¹). A uniform 2 °C warming produces only a modest downward shift (mean 4.17 t ha⁻¹), with substantial overlap with the historical distribution. Reducing rainfall by 20% shifts the distribution markedly downward (mean 2.55 t ha⁻¹; median 2.26 t ha⁻¹) and produces several very low-yield years. The combined scenario has the lowest central yield (mean 2.33 t ha⁻¹; median 1.99 t ha⁻¹) and the widest relative dispersion.

This graph is descriptive: overlap between boxplots is not a statistical test, and the four samples must not be treated as independent. Each point has a counterpart for the same harvest year in every scenario. The next analysis should therefore plot and test the within-year paired yield differences relative to HIST.

## 2. How does each climate scenario change yield within the same year?

The boxplots summarize four distributions but do not show whether a given climatic year responds consistently across scenarios. We therefore pair simulations by harvest year and calculate each scenario's yield difference from HIST. The absolute difference, $\Delta Y = Y_{scenario} - Y_{HIST}$, is expressed in t ha⁻¹. The relative difference, $100 \times \Delta Y / Y_{HIST}$, expresses the change as a percentage of that year's historical yield. Negative values indicate a yield loss and positive values a gain.

In [ ]:
comparison_scenarios = ["TPLUS2", "RAIN80", "TPLUS2_RAIN80"]
comparison_labels = {
    "TPLUS2": "+2 °C",
    "RAIN80": "Rain −20%",
    "TPLUS2_RAIN80": "+2 °C & Rain −20%",
}
comparison_colors = {
    "TPLUS2": "#F58518",
    "RAIN80": "#54A24B",
    "TPLUS2_RAIN80": "#E45756",
}
paired_years = sorted(reference_years)
if any(annual_yields["HIST"][year] == 0 for year in paired_years):
    raise ValueError("Relative differences are undefined because HIST contains a zero yield")

yield_differences = {}
relative_yield_differences = {}
for scenario in comparison_scenarios:
    absolute = [
        annual_yields[scenario][year] - annual_yields["HIST"][year]
        for year in paired_years
    ]
    relative = [
        100.0 * difference / annual_yields["HIST"][year]
        for year, difference in zip(paired_years, absolute)
    ]
    yield_differences[scenario] = absolute
    relative_yield_differences[scenario] = relative

print(f"{'Scenario vs HIST':<24} {'Mean ΔY':>10} {'Mean ΔY (%)':>13} {'Loss years':>12}")
for scenario in comparison_scenarios:
    absolute = yield_differences[scenario]
    relative = relative_yield_differences[scenario]
    loss_years = sum(value < 0 for value in absolute)
    print(
        f"{comparison_labels[scenario]:<24} {mean(absolute):10.2f} "
        f"{mean(relative):13.1f} {loss_years:>5}/{len(paired_years):<6}"
    )

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8.5), sharex=True)

for scenario in comparison_scenarios:
    label = comparison_labels[scenario]
    color = comparison_colors[scenario]
    axes[0].plot(
        paired_years, yield_differences[scenario],
        marker="o", markersize=4, linewidth=1.5, color=color,
        label=f"{label} (mean {mean(yield_differences[scenario]):+.2f})",
    )
    axes[1].plot(
        paired_years, relative_yield_differences[scenario],
        marker="o", markersize=4, linewidth=1.5, color=color,
        label=f"{label} (mean {mean(relative_yield_differences[scenario]):+.1f}%)",
    )

for axis in axes:
    axis.axhline(0, color="black", linewidth=1, linestyle="--")
    axis.grid(alpha=0.25)
    axis.legend(loc="best", fontsize=9)
axes[0].set_ylabel("Yield difference (t ha⁻¹)")
axes[0].set_title("Paired annual yield changes relative to HIST")
axes[1].set_ylabel("Relative yield change (%)")
axes[1].set_xlabel("Harvest year")
fig.tight_layout()
PAIRED_DIFFERENCES_PATH = (
    Path(data_dir).parent / "scripts" / "paired_annual_yield_differences.png"
)
PAIRED_DIFFERENCES_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(PAIRED_DIFFERENCES_PATH, dpi=200, bbox_inches="tight")
plt.show()
PAIRED_DIFFERENCES_PATH

### Interpretation

Pairing by year makes the rainfall signal much clearer than the boxplots alone. Rain −20% reduces yield in all 30 years, by about 1.81 t ha⁻¹ on average. The combined scenario also produces a loss in every year, averaging about 2.03 t ha⁻¹. By contrast, +2 °C alone has a smaller and heterogeneous response: the average difference is about −0.19 t ha⁻¹, with losses in 20 of 30 years and gains in the other 10.

The percentage panel must be interpreted carefully: a moderate absolute loss can become a very large percentage in a year when HIST yield is already low. These results remain descriptive. They show the direction and interannual consistency of responses, but they do not yet quantify uncertainty around the mean paired effects. The next step will estimate confidence intervals and perform paired statistical tests.

## 3. Are the mean paired yield changes statistically distinguishable from zero?

For each scenario, the statistical unit is the harvest year and the response is the within-year absolute difference from HIST (t ha⁻¹). We estimate the mean difference and its two-sided 95% Student confidence interval, then test $H_0: E(\Delta Y)=0$ with a paired t-test. A paired Wilcoxon signed-rank test is reported as a rank-based sensitivity analysis. Because three scenarios are compared with HIST, Holm-adjusted p-values control the family-wise error rate. The significance level is $\alpha=0.05$.

In [ ]:
from scipy import stats


def holm_adjust(p_values):
    """Return Holm-adjusted p-values in their original order."""
    order = sorted(range(len(p_values)), key=lambda index: p_values[index])
    adjusted = [0.0] * len(p_values)
    running_maximum = 0.0
    number_of_tests = len(p_values)
    for rank, index in enumerate(order):
        candidate = (number_of_tests - rank) * p_values[index]
        running_maximum = max(running_maximum, candidate)
        adjusted[index] = min(1.0, running_maximum)
    return adjusted


paired_test_results = []
for scenario in comparison_scenarios:
    differences = yield_differences[scenario]
    n = len(differences)
    mean_difference = mean(differences)
    standard_error = stats.sem(differences)
    margin = stats.t.ppf(0.975, df=n - 1) * standard_error
    t_result = stats.ttest_rel(
        [annual_yields[scenario][year] for year in paired_years],
        [annual_yields["HIST"][year] for year in paired_years],
    )
    wilcoxon_result = stats.wilcoxon(
        differences, alternative="two-sided", zero_method="wilcox"
    )
    paired_test_results.append({
        "scenario": scenario, "mean": mean_difference,
        "ci_low": mean_difference - margin,
        "ci_high": mean_difference + margin,
        "t_p": t_result.pvalue,
        "wilcoxon_p": wilcoxon_result.pvalue,
    })

t_p_adjusted = holm_adjust([result["t_p"] for result in paired_test_results])
w_p_adjusted = holm_adjust([result["wilcoxon_p"] for result in paired_test_results])
for result, t_adjusted, w_adjusted in zip(paired_test_results, t_p_adjusted, w_p_adjusted):
    result["t_p_holm"] = t_adjusted
    result["wilcoxon_p_holm"] = w_adjusted

print(f"{'Scenario vs HIST':<24} {'Mean ΔY [95% CI]':>27} {'paired t p':>12} {'Holm p':>11} {'Wilcoxon p':>13} {'Holm p':>11}")
for result in paired_test_results:
    interval = f"{result['mean']:+.2f} [{result['ci_low']:+.2f}, {result['ci_high']:+.2f}]"
    print(
        f"{comparison_labels[result['scenario']]:<24} {interval:>27} "
        f"{result['t_p']:12.3g} {result['t_p_holm']:11.3g} "
        f"{result['wilcoxon_p']:13.3g} {result['wilcoxon_p_holm']:11.3g}"
    )

In [ ]:
fig, axis = plt.subplots(figsize=(9.5, 5.2))
positions = list(range(len(paired_test_results)))
for position, result in zip(positions, paired_test_results):
    scenario = result["scenario"]
    significant = result["t_p_holm"] < 0.05
    axis.errorbar(
        result["mean"], position,
        xerr=[[result["mean"] - result["ci_low"]],
              [result["ci_high"] - result["mean"]]],
        fmt="o", markersize=8, capsize=5, linewidth=2,
        color=comparison_colors[scenario],
    )
    axis.text(
        result["ci_high"] + 0.08, position,
        "Holm p = " + format(result["t_p_holm"], ".3g"),
        ha="left", va="center", fontsize=9,
        fontweight="bold" if significant else "normal",
    )

axis.axvline(0, color="black", linewidth=1, linestyle="--", label="HIST reference")
axis.set_yticks(positions, [comparison_labels[r["scenario"]] for r in paired_test_results])
axis.invert_yaxis()
axis.set_xlabel("Mean paired yield difference from HIST (t ha⁻¹)")
axis.set_title("Mean climate-scenario effects with 95% confidence intervals")
axis.set_xlim(
    min(result["ci_low"] for result in paired_test_results) - 0.15,
    max(0, max(result["ci_high"] for result in paired_test_results)) + 0.55,
)
axis.grid(axis="x", alpha=0.25)
axis.legend(loc="lower right", fontsize=9)
fig.tight_layout()
PAIRED_TESTS_PATH = Path(data_dir).parent / "scripts" / "paired_yield_tests_ci95.png"
PAIRED_TESTS_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(PAIRED_TESTS_PATH, dpi=200, bbox_inches="tight")
plt.show()
PAIRED_TESTS_PATH

### Interpretation

The +2 °C scenario has an estimated mean effect of −0.19 t ha⁻¹, but its 95% confidence interval includes zero (approximately −0.46 to +0.08 t ha⁻¹). Its paired t-test and Wilcoxon test are therefore not significant at the 5% level. These data do not provide sufficient evidence of a systematic temperature-only effect, which is not equivalent to demonstrating that warming has no effect.

Rain −20% has an estimated mean loss of −1.81 t ha⁻¹ (95% CI approximately −2.18 to −1.43), while the combined scenario loses −2.03 t ha⁻¹ (approximately −2.45 to −1.61). Both effects remain highly significant after Holm correction with either paired test. The agreement between the t-test and Wilcoxon conclusions indicates that these findings are not driven solely by the parametric assumption.

The confidence intervals and tests treat annual paired differences as independent observations. Because these are successive simulations and climate years may be temporally correlated, this assumption should be checked before making a formal inferential claim. A later step can examine autocorrelation and, if necessary, use a block bootstrap or time-series model. These p-values describe variability among the 30 simulated weather years; they do not represent uncertainty in DSSAT parameters, model structure, or the imposed climate perturbations.

## 4. Does climate change alter crop phenology?

Yield is an endpoint, so we first examine a possible mechanism: changes in crop-calendar duration. DSSAT dates are converted from `YYYYDDD` to calendar dates. For every scenario and harvest year, we calculate days from planting to emergence, anthesis and maturity, plus the anthesis-to-maturity interval. Shorter durations under warming would indicate faster development and a modified period of exposure to rainfall and high temperature.

In [ ]:
from datetime import datetime, timedelta


def read_annual_rows(path):
    with path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    return {int(float(row["HYEAR"])): row for row in rows}


def dssat_date(value):
    return datetime.strptime(str(int(float(value))), "%Y%j").date()


annual_rows = {scenario: read_annual_rows(path) for scenario, path in result_files.items()}
phenology_metrics = {
    "planting_to_emergence": "Planting → emergence",
    "planting_to_anthesis": "Planting → anthesis",
    "planting_to_maturity": "Planting → maturity",
    "anthesis_to_maturity": "Anthesis → maturity",
}
phenology = {scenario: {metric: {} for metric in phenology_metrics} for scenario in scenario_order}
for scenario in scenario_order:
    if set(annual_rows[scenario]) != reference_years:
        raise ValueError(f"{scenario} has incomplete phenology years")
    for year, row in annual_rows[scenario].items():
        planting = dssat_date(row["Planting"])
        emergence = dssat_date(row["Emergence"])
        anthesis = dssat_date(row["Ant"])
        maturity = dssat_date(row["Mat"])
        phenology[scenario]["planting_to_emergence"][year] = (emergence - planting).days
        phenology[scenario]["planting_to_anthesis"][year] = (anthesis - planting).days
        phenology[scenario]["planting_to_maturity"][year] = (maturity - planting).days
        phenology[scenario]["anthesis_to_maturity"][year] = (maturity - anthesis).days

print(f"{'Scenario':<22}" + ''.join(f"{label:>24}" for label in phenology_metrics.values()))
for scenario, label in zip(scenario_order, scenario_labels):
    averages = [mean(phenology[scenario][metric].values()) for metric in phenology_metrics]
    print(f"{label:<22}" + ''.join(f"{value:24.1f}" for value in averages))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for axis, (metric, metric_label) in zip(axes.flat, phenology_metrics.items()):
    data = [[phenology[scenario][metric][year] for year in paired_years] for scenario in scenario_order]
    boxes = axis.boxplot(data, patch_artist=True, showmeans=True, tick_labels=scenario_labels)
    for patch, color in zip(boxes["boxes"], scenario_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    axis.set_title(metric_label)
    axis.set_ylabel("Duration (days)")
    axis.grid(axis="y", alpha=0.25)
    axis.tick_params(axis="x", rotation=18)
fig.suptitle("Simulated wheat phenology under the four climate scenarios", y=1.01)
fig.tight_layout()
PHENOLOGY_PATH = Path(data_dir).parent / "scripts" / "phenology_scenarios.png"
PHENOLOGY_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(PHENOLOGY_PATH, dpi=200, bbox_inches="tight")
plt.show()
PHENOLOGY_PATH

### Interpretation

Warming advances development: mean planting-to-anthesis duration decreases from 107.2 days under HIST to 96.8 days under +2 °C, and planting-to-maturity decreases from 163.0 to 151.7 days. Similar shortening occurs in the combined scenario. Rainfall reduction alone changes anthesis and maturity duration very little, but delays emergence on average (13.6 instead of 8.7 days), consistent with drier conditions around establishment. The anthesis-to-maturity interval changes less than the full cycle. Consequently, thermal exposure must be calculated within each scenario's own phenological dates rather than over one fixed calendar window.

## 5. When during development does DSSAT simulate water stress?

DSSAT's `OVERVIEW.OUT` reports phase-specific mean water stress for photosynthesis and growth on a 0–1 scale, where 0 means no stress and 1 maximum stress. We extract the photosynthesis stress for five development phases from each of the 30 runs. This is a modelled stress response, unlike rainfall or evapotranspiration alone, which are water-balance quantities. The phase values should be interpreted alongside phase duration and rainfall.

In [ ]:
WATER_STRESS_PHASES = [
    "Germinate - Term Spklt",
    "Term Spklt - End Veg",
    "End Veg - End Ear Gr",
    "End Ear Gr - Beg Gr Fil",
    "Beg Gr Fil - End Gr Fil",
]
WATER_STRESS_PHASE_LABELS = [
    "Germination–terminal spikelet",
    "Terminal spikelet–end vegetative",
    "End vegetative–end ear growth",
    "End ear growth–start grain filling",
    "Grain filling",
]


def latest_overview(directory):
    candidates = list((directory / "temp").glob("**/OVERVIEW.OUT"))
    if not candidates:
        raise FileNotFoundError(f"No OVERVIEW.OUT found below {directory / 'temp'}")
    return max(candidates, key=lambda path: path.stat().st_mtime)


def read_phase_water_stress(path, years):
    records = []
    run_number = None
    with path.open(encoding="utf-8", errors="replace") as handle:
        for raw_line in handle:
            if raw_line.startswith("*RUN"):
                run_number = int(raw_line.split()[1])
                continue
            normalized = " ".join(raw_line.split())
            for phase in WATER_STRESS_PHASES:
                if normalized.startswith(phase + " "):
                    values = [float(value) for value in normalized[len(phase):].split()]
                    if len(values) != 13 or run_number is None:
                        raise ValueError(f"Unexpected stress row in {path}: {raw_line}")
                    records.append({
                        "run": run_number, "phase": phase, "days": values[0],
                        "tmax": values[1], "tmin": values[2], "rain": values[5],
                        "transpiration": values[6],
                        "water_photo_stress": values[7],
                        "water_growth_stress": values[8],
                    })
                    break
    run_numbers = sorted({record["run"] for record in records})
    if len(run_numbers) != len(years):
        raise ValueError(f"{path} contains {len(run_numbers)} runs for {len(years)} years")
    run_to_year = dict(zip(run_numbers, sorted(years)))
    for record in records:
        record["year"] = run_to_year[record["run"]]
    if len(records) != len(years) * len(WATER_STRESS_PHASES):
        raise ValueError(f"Incomplete phase-stress data in {path}")
    return records


overview_files = {scenario: latest_overview(directory) for scenario, directory in SCENARIO_OUTPUTS.items()}
water_stress_records = {
    scenario: read_phase_water_stress(path, reference_years)
    for scenario, path in overview_files.items()
}
mean_phase_water_stress = []
for scenario in scenario_order:
    scenario_means = []
    for phase in WATER_STRESS_PHASES:
        values = [r["water_photo_stress"] for r in water_stress_records[scenario] if r["phase"] == phase]
        scenario_means.append(mean(values))
    mean_phase_water_stress.append(scenario_means)

print("Mean DSSAT water stress for photosynthesis (0=no stress; 1=maximum)")
for scenario, label, values in zip(scenario_order, scenario_labels, mean_phase_water_stress):
    print(f"{label:<22}" + " ".join(f"{value:.3f}" for value in values))

In [ ]:
fig, axis = plt.subplots(figsize=(12, 4.8))
image = axis.imshow(mean_phase_water_stress, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
axis.set_xticks(range(len(WATER_STRESS_PHASE_LABELS)), WATER_STRESS_PHASE_LABELS, rotation=22, ha="right")
axis.set_yticks(range(len(scenario_labels)), scenario_labels)
for row_index, row in enumerate(mean_phase_water_stress):
    for column_index, value in enumerate(row):
        axis.text(column_index, row_index, f"{value:.2f}", ha="center", va="center",
                  color="white" if value > 0.58 else "black", fontweight="bold")
colorbar = fig.colorbar(image, ax=axis, pad=0.02)
colorbar.set_label("Mean photosynthesis water stress (0–1)")
axis.set_title("Phase-specific DSSAT water stress across 30 growing seasons")
fig.tight_layout()
WATER_STRESS_PATH = Path(data_dir).parent / "scripts" / "water_stress_by_phase.png"
WATER_STRESS_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(WATER_STRESS_PATH, dpi=200, bbox_inches="tight")
plt.show()
WATER_STRESS_PATH

### Interpretation

DSSAT simulates the strongest average water limitation from terminal spikelet through the beginning of grain filling. Reducing rainfall increases early stress markedly: mean stress from germination to terminal spikelet rises from about 0.10 under HIST to 0.20 under RAIN80, and terminal-spikelet-to-end-vegetative stress rises from about 0.47 to 0.61. The combined scenario reaches approximately 0.23 and 0.66 in these phases. Warming changes both phase timing and rainfall encountered by the crop, so its phase-specific stress response is not a uniform increase. These values are averages: distributions among years should be inspected before generalizing, and the index describes DSSAT's simulated limitation rather than directly observed plant stress.

## 6. Does warming increase reproductive heat exposure?

The current successive outputs do not provide a dedicated DSSAT heat-stress index. We therefore calculate transparent exposure indicators from daily `RAClimateD.tmax`, using each scenario's own anthesis and maturity dates: the percentage of reproductive days with Tmax ≥30 °C, the percentage with Tmax ≥35 °C, and the number of Tmax ≥30 °C days in a ±7-day window around anthesis. These thresholds are illustrative exposure metrics, not calibrated biological damage thresholds for cultivar Karim.

In [ ]:
import sqlite3

SCENARIO_DATABASES = {scenario: Path(data_dir) / f"MasterInput_{scenario}.db" for scenario in scenario_order}
thermal_exposure = {scenario: {} for scenario in scenario_order}
for scenario in scenario_order:
    database_path = SCENARIO_DATABASES[scenario]
    connection = sqlite3.connect(f"file:{database_path}?mode=ro", uri=True)
    try:
        for year, row in annual_rows[scenario].items():
            anthesis = dssat_date(row["Ant"])
            maturity = dssat_date(row["Mat"])
            reproductive_tmax = [value[0] for value in connection.execute(
                "SELECT tmax FROM RAClimateD WHERE date(w_date) BETWEEN ? AND ? ORDER BY w_date",
                (anthesis.isoformat(), maturity.isoformat()),
            )]
            anthesis_window_tmax = [value[0] for value in connection.execute(
                "SELECT tmax FROM RAClimateD WHERE date(w_date) BETWEEN ? AND ? ORDER BY w_date",
                ((anthesis - timedelta(days=7)).isoformat(), (anthesis + timedelta(days=7)).isoformat()),
            )]
            if not reproductive_tmax or len(anthesis_window_tmax) != 15:
                raise ValueError(f"Incomplete daily climate for {scenario}, harvest year {year}")
            thermal_exposure[scenario][year] = {
                "hot30_fraction": 100 * sum(value >= 30 for value in reproductive_tmax) / len(reproductive_tmax),
                "hot35_fraction": 100 * sum(value >= 35 for value in reproductive_tmax) / len(reproductive_tmax),
                "anthesis_hot30_days": sum(value >= 30 for value in anthesis_window_tmax),
                "mean_reproductive_tmax": mean(reproductive_tmax),
            }
    finally:
        connection.close()

print(f"{'Scenario':<22} {'Reproductive Tmax':>20} {'Days ≥30 °C (%)':>18} {'Days ≥35 °C (%)':>18} {'±7 d anthesis ≥30':>20}")
for scenario, label in zip(scenario_order, scenario_labels):
    records = thermal_exposure[scenario].values()
    print(f"{label:<22} {mean(r['mean_reproductive_tmax'] for r in records):20.1f} "
          f"{mean(r['hot30_fraction'] for r in records):18.1f} "
          f"{mean(r['hot35_fraction'] for r in records):18.1f} "
          f"{mean(r['anthesis_hot30_days'] for r in records):20.1f}")

In [ ]:
thermal_metrics = {
    "hot30_fraction": "Reproductive days with Tmax ≥30 °C (%)",
    "hot35_fraction": "Reproductive days with Tmax ≥35 °C (%)",
    "anthesis_hot30_days": "Days with Tmax ≥30 °C around anthesis (±7 d)",
}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for axis, (metric, label) in zip(axes, thermal_metrics.items()):
    data = [[thermal_exposure[scenario][year][metric] for year in paired_years] for scenario in scenario_order]
    boxes = axis.boxplot(data, patch_artist=True, showmeans=True, tick_labels=scenario_labels)
    for patch, color in zip(boxes["boxes"], scenario_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    axis.set_title(label, fontsize=10)
    axis.grid(axis="y", alpha=0.25)
    axis.tick_params(axis="x", rotation=24)
fig.suptitle("Thermal exposure within scenario-specific reproductive periods", y=1.02)
fig.tight_layout()
THERMAL_EXPOSURE_PATH = Path(data_dir).parent / "scripts" / "thermal_exposure_scenarios.png"
THERMAL_EXPOSURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(THERMAL_EXPOSURE_PATH, dpi=200, bbox_inches="tight")
plt.show()
THERMAL_EXPOSURE_PATH

### Interpretation

The +2 °C perturbation does not translate into a simple +2 °C increase in temperature experienced during reproduction. Warming advances anthesis and maturity by roughly 10–11 days, moving the reproductive period toward earlier and generally cooler calendar dates. In the current simulations, this phenological escape partly compensates for the imposed warming, so mean reproductive Tmax and the frequency of hot days change less than expected from the weather perturbation alone. This helps explain why the temperature-only yield response is heterogeneous. The indicators quantify exposure, not damage: demonstrating heat stress would require a justified crop-stage-specific response function or an explicit DSSAT heat-stress output.